<a href="https://colab.research.google.com/github/SamShinwari/NLP/blob/master/pashto_stemming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

In [ ]:
# -*- coding: utf-8 -*-

# =========================================================
# PASHTO LANGUAGE RULE-BASED STEMMER
# Google Colab Version
# =========================================================

# Install pandas if needed
# !pip install pandas

import pandas as pd
import re

# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive

drive.mount('/content/drive')

# =========================================================
# LOAD DATASET FROM GOOGLE DRIVE
# =========================================================

file_path = '/content/drive/MyDrive/Colab Notebooks/pashto_dataset.csv'

df = pd.read_csv(file_path)

print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)

# =========================================================
# REMOVE NULL VALUES & DUPLICATES
# =========================================================

df = df.dropna(subset=["desc"])

df = df.drop_duplicates(subset=["desc"])

df = df.reset_index(drop=True)

print("After Cleaning Shape:", df.shape)

# =========================================================
# PASHTO STEMMER CLASS
# =========================================================

class PashtoStemmer:

    def __init__(self):

        # -------------------------------------------------
        # RULE 1 SUFFIXES
        # -------------------------------------------------

        self.rule1_suffixes = [

            "ستان",
            "تون",
            "ونو",
            "ان",
            "و",
            "ور",
            "ناک",
            "جن",
            "وان",
            "وال",
            "تابه"
        ]

        # -------------------------------------------------
        # RULE 5 PREFIXES
        # -------------------------------------------------

        self.rule5_prefixes = [

            "نه",
            "نا",
            "م",
            "بد",
            "نیک",
            "تش",
            "تل",
            "تور",
            "تک",
            "ډیر"
        ]

        # -------------------------------------------------
        # RULE 9 PREFIXES
        # -------------------------------------------------

        self.rule9_prefixes = [

            "نه",
            "نا",
            "نیک"
        ]

    # =====================================================
    # NORMALIZATION
    # =====================================================

    def normalize(self, word):

        punctuations = r"""!()-[]{};:'"\,<>./?@#$%^&*_~،؛؟"""

        for p in punctuations:

            word = word.replace(p, "")

        return word.strip()

    # =====================================================
    # RULE 1
    # Remove suffixes
    # =====================================================

    def rule1(self, word):

        if len(word) >= 5:

            for suf in self.rule1_suffixes:

                if word.endswith(suf):

                    word = word[:-len(suf)]

                    break

        return word

    # =====================================================
    # RULE 2
    # Replace ي with ه
    # =====================================================

    def rule2(self, word):

        if len(word) >= 4 and word.endswith("ي"):

            word = word[:-1] + "ه"

        return word

    # =====================================================
    # RULE 3
    # Remove من
    # =====================================================

    def rule3(self, word):

        if len(word) >= 5 and word.endswith("من"):

            word = word[:-2]

        return word

    # =====================================================
    # RULE 4
    # Remove یز and add ه
    # =====================================================

    def rule4(self, word):

        if len(word) >= 5 and word.endswith("یز"):

            word = word[:-2] + "ه"

        return word

    # =====================================================
    # RULE 5
    # Remove prefixes
    # =====================================================

    def rule5(self, word):

        if len(word) > 3:

            for pre in self.rule5_prefixes:

                if word.startswith(pre):

                    word = word[len(pre):]

                    break

        return word

    # =====================================================
    # RULE 6
    # Remove ال
    # =====================================================

    def rule6(self, word):

        if len(word) >= 5 and word.startswith("ال"):

            word = word[2:]

        return word

    # =====================================================
    # RULE 7
    # Remove وران
    # =====================================================

    def rule7(self, word):

        if len(word) >= 5 and word.startswith("وران"):

            word = word[4:]

        return word

    # =====================================================
    # RULE 8
    # Remove repeated consecutive words
    # =====================================================

    def rule8(self, words):

        cleaned_words = []

        for word in words:

            if len(cleaned_words) == 0:

                cleaned_words.append(word)

            elif cleaned_words[-1] != word:

                cleaned_words.append(word)

        return cleaned_words

    # =====================================================
    # RULE 9
    # Remove نه نا نیک
    # =====================================================

    def rule9(self, word):

        if len(word) > 3:

            for pre in self.rule9_prefixes:

                if word.startswith(pre):

                    word = word[len(pre):]

                    break

        return word

    # =====================================================
    # STEM SINGLE WORD
    # =====================================================

    def stem_word(self, word):

        word = self.normalize(word)

        word = self.rule1(word)

        word = self.rule2(word)

        word = self.rule3(word)

        word = self.rule4(word)

        word = self.rule5(word)

        word = self.rule6(word)

        word = self.rule7(word)

        word = self.rule9(word)

        return word

    # =====================================================
    # STEM COMPLETE TEXT
    # =====================================================

    def stem_text(self, text):

        if not isinstance(text, str):

            return ""

        words = text.split()

        stemmed_words = []

        for word in words:

            root = self.stem_word(word)

            stemmed_words.append(root)

        # Apply Rule 8
        stemmed_words = self.rule8(stemmed_words)

        return " ".join(stemmed_words)


# =========================================================
# CLEAN TEXT FUNCTION
# =========================================================

def clean_text(text):

    if not isinstance(text, str):

        return ""

    # Remove URLs
    text = re.sub(r'https?:\/\/\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\S+', '', text)

    # Remove hashtags
    text = re.sub(r'#\S+', '', text)

    # Remove English text
    text = re.sub(r'[A-Za-z]', '', text)

    # Remove digits
    text = re.sub(r'[۰۱۲۳۴۵۶۷۸۹0-9]', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# =========================================================
# APPLY CLEANING
# =========================================================

df["clean_text"] = df["desc"].apply(clean_text)

# =========================================================
# APPLY STEMMING
# =========================================================

stemmer = PashtoStemmer()

df["stemmed_text"] = df["clean_text"].apply(
    stemmer.stem_text
)

# =========================================================
# SHOW RESULTS
# =========================================================

print("\n")
print("=" * 60)
print("PASHTO STEMMING RESULTS")
print("=" * 60)

for i in range(min(10, len(df))):

    print("\nOriginal Text:")
    print(df["desc"][i])

    print("\nClean Text:")
    print(df["clean_text"][i])

    print("\nStemmed Text:")
    print(df["stemmed_text"][i])

    print("-" * 60)

# =========================================================
# SAVE OUTPUT TO GOOGLE DRIVE
# =========================================================

output_path = '/content/drive/MyDrive/Colab Notebooks/pashto_stemmed_output.csv'

df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n")
print("=" * 60)
print("OUTPUT SAVED SUCCESSFULLY")
print("Saved File:")
print(output_path)
print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Loaded Successfully
Dataset Shape: (2922, 3)
After Cleaning Shape: (2743, 3)


PASHTO STEMMING RESULTS

Original Text:
له دې اعلان وروسته د دغه هېواد لر او بر لاریونونه هم وشول او خلکو د برېښنا بېلونه هم وسوځول.,ځينو خلکو ته ویل شوي چې بېلونه یې پر وخت نه دي ورکړي نو ځکه باید اوس جریمه هم ورکړي. پاکستان د انرژۍ د کمي له سخت بحران سره مخامخ دی او د پېټرول او د برېښنا بيې یې ريکارډ کچي ته لوړې شوي.,خو لاملونه یې څه دي، په پېښور کي زموږ همکار رحمان لله په خپل راپور کي ورته کتنه کړې.]

Clean Text:
له دې اعلان وروسته د دغه هېواد لر او بر لاریونونه هم وشول او خلکو د برېښنا بېلونه هم وسوځول ځينو خلکو ته ویل شوي چې بېلونه یې پر وخت نه دي ورکړي نو ځکه باید اوس جریمه هم ورکړي پاکستان د انرژۍ د کمي له سخت بحران سره مخامخ دی او د پېټرول او د برېښنا بيې یې ريکارډ کچي ته لوړې شوي خو لاملونه یې څه دي په پېښور کي زموږ همکار رحمان لله په خپل راپور کي ورته کتنه کړې

St